In [76]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm, tqdm_notebook
import csv
from datetime import datetime

In [77]:
VAL_SPLIT = 0.2
CONTEXT_LENGTH = 256
DATA_STRIDE = 1
EMB_SIZE = 256
BATCH_SIZE = 6
VOCAB_SIZE = None
RESULTS_FOLDER = "./train_runs"

TOTAL_EPOCHS = 10
DEVICE = "mps"

In [ ]:
def write_rows(dict_data: dict, model_name=""):
    f_name = f"{RESULTS_FOLDER}/{model_name}__{(datetime.now()).strftime("%y%m%d%H%M")}.csv"
    headers = list(dict_data[0].keys())
    with open(f_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(dict_data)
        
# write_rows(results, model_name="base_model_attenion_head")

### Data Loading

In [79]:
with open("the-verdict-ru.txt", "r") as f:
    data = f.read()
    
data

'Main hamesha Jack Gisburn ko kuch satahi sa zaheen fankar samajhta tha--halaanke aadmi woh khaasa achha tha--is liye mujhe yeh sun kar koi khaas hairat nahin hui ke apni shohrat ke urooj par us ne musavviri chhor di, ek ameer bewa se shaadi kar li, aur Riviera ke ek villa mein bas gaya. (Waise mera zyada khayal tha ke woh Rome ya Florence mein ja basega.)\n\n"Apni shohrat ke urooj par"--auratein to yahi kehti thin. Mujhe ab bhi Mrs. Gideon Thwing ki awaaz sunai deti hai--Chicago mein sab se aakhir mein unhon ne us se apni tasveer banwai thi--jo us ke is na-samajh mein aane wale tark-e-fann par afsos kar rahi thin. "Zahir hai, is se meri tasveer ki qeemat bohat barh jayegi; lekin main us ki fikr nahin karti, Mr. Rickham--mujhe to bas Aarrt ke nuqsan ki fikr hai." Mrs. Thwing ke honton par angrezi lafz "Art" ke _r_ yun barhte chale jaate the, jaise be-intiha door tak lage aainon mein un ka aks baar baar nazar aa raha ho. Aur maatam karne wali sirf Mrs. Thwing jaisi khawateen hi nahin th

In [80]:
tokenizer_gpt2 = tiktoken.get_encoding("gpt2")
VOCAB_SIZE = tokenizer_gpt2.n_vocab

In [81]:
tokenized_data = tokenizer_gpt2.encode(data)
len(tokenized_data), len(data), tokenizer_gpt2.max_token_value # English Version: 5145, 20479, 50256 | RU Version: 10854, 26773, 50256

(10854, 26773, 50256)

In [82]:
class VerdictDataset(Dataset):
    def __init__(
        self,
        tokenized_data,
        context_len=CONTEXT_LENGTH,
        stride=DATA_STRIDE,
    ):
        super().__init__()
        self.inputs, self.targets = [], []
        for i in range(0, len(tokenized_data) - context_len, stride):
            inputs = tokenized_data[i : context_len + i]
            targets = tokenized_data[i + 1 : context_len + i + 1]
            # uncomment to check the logs
            # print(inputs, targets)
            # print(tokenizer_gpt2.decode(inputs), "---", tokenizer_gpt2.decode(targets))
            self.inputs.append(inputs)
            self.targets.append(targets)
        self.inputs = torch.tensor(self.inputs)
        self.targets = torch.tensor(self.targets)

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.targets[index]

In [83]:
train_split = len(tokenized_data) - (len(tokenized_data) * VAL_SPLIT)
train_end = int(train_split)
train_end

verdict_dataset_train = VerdictDataset(tokenized_data=tokenized_data[0:train_end])
verdict_dataset_valid = VerdictDataset(tokenized_data=tokenized_data[train_end:])

train_ds_loader = DataLoader(
    verdict_dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)
val_ds_loader = DataLoader(verdict_dataset_valid, batch_size=BATCH_SIZE, num_workers=0)

In [84]:
# English Version: 5145, 3860, 773 | RU Version: 10854, 8427, 1915
len(tokenized_data), len(verdict_dataset_train), len(verdict_dataset_valid) 

(10854, 8427, 1915)

In [85]:
for i, (ip, tar) in enumerate(val_ds_loader):
    # print(ip.shape, ip.tolist()[0])
    # break
    print(tokenizer_gpt2.decode(ip.tolist()[0]), "---", tokenizer_gpt2.decode(tar.tolist()[0]))
    break

u hone lagin--main ghabra gaya aur mera aitmaad dagmagane laga.

"Ek baar jab maine nazar uthayi to mujhe us ki chhoti, surmai daarhi ke peechhe ek muskurahat si nazar aayi--jaise raaz us ke paas ho, aur woh mujh se use chhupa kar lutf le raha ho. Is se main aur bhi chir gaya. Raaz? Aray, mere paas to aisa raaz tha jo us ke bees raazon par bhaari tha! Main ghusse se canvas par toot para aur apni numaishi maharat ke kuch kartab aazmaye. Magar woh mere kaam na aaye, bikhar gaye. Maine dekha ke woh chamak-damak wale hisson ko dekh hi nahin raha tha--main us ki tawajjoh hata nahin sakta tha; us ki nazar to bas un ke darmiyan ke mushkil hisson par jami h ---  hone lagin--main ghabra gaya aur mera aitmaad dagmagane laga.

"Ek baar jab maine nazar uthayi to mujhe us ki chhoti, surmai daarhi ke peechhe ek muskurahat si nazar aayi--jaise raaz us ke paas ho, aur woh mujh se use chhupa kar lutf le raha ho. Is se main aur bhi chir gaya. Raaz? Aray, mere paas to aisa raaz tha jo us ke bees raazon p

### Basic Model Training

In [10]:
# Experiment with what happens with the data
for i, (ip, tar) in enumerate(val_ds_loader):
    print(ip.shape)
    ip1 = torch.nn.Embedding(tokenizer_gpt2.max_token_value, EMB_SIZE)(ip)
    print(ip1.shape)
    ip2 = torch.nn.Linear(128, 256)(ip1)
    print(ip2.shape)
    ip3 = torch.nn.GELU()(ip2)
    print(ip3.shape)
    ip4 = torch.nn.Linear(256, tokenizer_gpt2.n_vocab)(ip3)
    print(ip4.shape)
    # break
    # print(tokenizer_gpt2.decode(ip.tolist()[0]), "---", tokenizer_gpt2.decode(tar.tolist()[0]))
    break

torch.Size([4, 256])
torch.Size([4, 256, 128])
torch.Size([4, 256, 256])
torch.Size([4, 256, 256])
torch.Size([4, 256, 50257])


In [11]:
class BasicModel(torch.nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        emb_size=EMB_SIZE,
        hidden_dim=256,
    ):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Embedding(vocab_size, emb_size),
            torch.nn.Linear(emb_size, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, VOCAB_SIZE),
        )

    def forward(self, x):
        return self.layers(x)
    
basic_model = BasicModel()

In [ ]:
optim = torch.optim.AdamW(basic_model.parameters(), lr=0.01)
basic_model.to(DEVICE)
results = []
for epoch in range(TOTAL_EPOCHS):
    progress_bar = tqdm(
        train_ds_loader,
        desc=f"Epoch {epoch + 1}/{TOTAL_EPOCHS}",
    )
    for ip, tar in progress_bar:
        optim.zero_grad()
        # print(ip.shape, tar.shape)
        out_logits = basic_model(ip.to(DEVICE))
        loss = torch.nn.functional.cross_entropy(
            out_logits.transpose(1, 2), tar.to(DEVICE)
        )
        # print(loss)
        loss.backward()
        optim.step()
        progress_bar.set_postfix(loss=f"{loss:.4f}")
    results.append({"epoch": epoch, "loss": loss})

Epoch 10/2: 100%|██████████| 2106/2106 [03:00<00:00, 11.68it/s, loss=2.1646]


In [29]:
# English version loss was around 1.8 - 1.7
# RU version loss was around 2.3 - 2.1
write_rows(results, model_name="basic_model")

In [ ]:
test_text = "Suno Dost Shah Nawaz "
test_text_encoded = tokenizer_gpt2.encode(test_text)
basic_model.eval()
with torch.no_grad():
    for _ in range(20):
        out_logits = basic_model(torch.tensor(test_text_encoded).to("mps"))
        # last token as next word
        out_logits = out_logits[-1]
        out_logits = torch.softmax(out_logits, dim=-1)
        next_token = torch.argmax(out_logits)
        next_token = [next_token.item()]
        test_text_encoded += next_token
        next_token_decoded = tokenizer_gpt2.decode(next_token)
        test_text += next_token_decoded
print(test_text)

### Basic Model + Positional Encoding

In [49]:
# Experiment with what happens with the data
for i, (ip, tar) in enumerate(val_ds_loader):
    print(ip.shape)
    ip1 = torch.nn.Embedding(tokenizer_gpt2.max_token_value, EMB_SIZE)(ip)
    print(ip1.shape)
    pos_emb = torch.nn.Embedding(tokenizer_gpt2.max_token_value, EMB_SIZE)
    pos_emb_to_add = pos_emb(torch.arange(0, ip1.shape[1]))
    ip11 = ip1 + pos_emb_to_add
    print(ip1.shape)
    ip2 = torch.nn.Linear(128, 256)(ip11)
    print(ip2.shape)
    ip3 = torch.nn.GELU()(ip2)
    print(ip3.shape)
    ip4 = torch.nn.Linear(256, tokenizer_gpt2.n_vocab)(ip3)
    print(ip4.shape)
    # break
    # print(tokenizer_gpt2.decode(ip.tolist()[0]), "---", tokenizer_gpt2.decode(tar.tolist()[0]))
    break

torch.Size([4, 256])
torch.Size([4, 256, 128])
torch.Size([4, 256, 128])
torch.Size([4, 256, 256])
torch.Size([4, 256, 256])
torch.Size([4, 256, 50257])


In [66]:
class BasicModelWithPosEmb(torch.nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        emb_size=EMB_SIZE,
        hidden_dim=256,
    ):
        super().__init__()
        self.emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.pos_emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(emb_size, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, VOCAB_SIZE),
        )

    def forward(self, x):
        x = self.emb_layer(x)
        x_pos = self.pos_emb_layer(torch.arange(0, x.shape[1]).to(DEVICE))
        x += x_pos
        return self.layers(x)


basic_model_with_pos = BasicModelWithPosEmb()

In [69]:
optim = torch.optim.AdamW(basic_model_with_pos.parameters(), lr=0.01)
basic_model_with_pos.to(DEVICE)
results = []
for epoch in range(TOTAL_EPOCHS):
    progress_bar = tqdm(
        train_ds_loader,
        desc=f"Epoch {epoch + 1}/{TOTAL_EPOCHS}",
    )
    for ip, tar in progress_bar:
        optim.zero_grad()
        # print(ip.shape, tar.shape)
        out_logits = basic_model_with_pos(ip.to(DEVICE))
        loss = torch.nn.functional.cross_entropy(
            out_logits.transpose(1, 2), tar.to(DEVICE)
        )
        # print(loss)
        loss.backward()
        optim.step()
        progress_bar.set_postfix(loss=f"{loss:.4f}")
    results.append({"epoch": epoch, "loss": loss.item()})

Epoch 10/10: 100%|██████████| 1404/1404 [02:59<00:00,  7.82it/s, loss=2.1312]


In [73]:
test_text = "Suno Dost, Adeel"
test_text_encoded = tokenizer_gpt2.encode(test_text)
basic_model_with_pos.eval()
with torch.no_grad():
    for _ in range(20):
        out_logits = basic_model_with_pos(torch.tensor([test_text_encoded]).to(DEVICE))
        # last token as next word
        out_logits = out_logits.squeeze()[-1]
        out_logits = torch.softmax(out_logits, dim=-1)
        next_token = torch.argmax(out_logits)
        next_token = [next_token.item()]
        test_text_encoded += next_token
        next_token_decoded = tokenizer_gpt2.decode(next_token)
        test_text += next_token_decoded
print(test_text)

Suno Dost, Adeel rangon ke saath aap ko dekhne ke saath aap ko dekhne


### + Attention Head 

In [92]:
class AttentionHead(torch.nn.Module):
    def __init__(self, emb_size, hidden_dims, qkv_bias=False):
        super().__init__()
        self.WQ = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)
        self.WK = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)
        self.WV = torch.nn.Linear(emb_size, hidden_dims, bias=qkv_bias)

    def forward(self, x):
        Q = self.WQ(x)
        K = self.WK(x)
        V = self.WV(x)
        context_vector = (
            torch.softmax((Q @ K.transpose(1, 2)) / x.shape[-1] ** 0.5, dim=-1) @ V
        )
        return context_vector

In [93]:
# Experiment with what happens with the data
for i, (ip, tar) in enumerate(val_ds_loader):
    print(ip.shape)
    ip1 = torch.nn.Embedding(tokenizer_gpt2.max_token_value, EMB_SIZE)(ip)
    print(ip1.shape)
    pos_emb = torch.nn.Embedding(tokenizer_gpt2.max_token_value, EMB_SIZE)
    pos_emb_to_add = pos_emb(torch.arange(0, ip1.shape[1]))
    ip11 = ip1 + pos_emb_to_add
    print(ip1.shape)
    ip12 = AttentionHead(EMB_SIZE, EMB_SIZE)(ip11)
    ip2 = torch.nn.Linear(EMB_SIZE, EMB_SIZE)(ip12)
    print(ip2.shape)
    ip3 = torch.nn.GELU()(ip2)
    print(ip3.shape)
    ip4 = torch.nn.Linear(EMB_SIZE, tokenizer_gpt2.n_vocab)(ip3)
    print(ip4.shape)
    # break
    # print(tokenizer_gpt2.decode(ip.tolist()[0]), "---", tokenizer_gpt2.decode(tar.tolist()[0]))
    break

torch.Size([6, 256])
torch.Size([6, 256, 256])
torch.Size([6, 256, 256])
torch.Size([6, 256, 256])
torch.Size([6, 256, 256])
torch.Size([6, 256, 50257])


In [95]:
class BasicModelWithPosEmbAttenHead(torch.nn.Module):
    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        emb_size=EMB_SIZE,
        hidden_dim=256,
    ):
        super().__init__()
        self.emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.pos_emb_layer = torch.nn.Embedding(vocab_size, emb_size)
        self.attention_head = AttentionHead(hidden_dim, hidden_dim)
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(hidden_dim, hidden_dim),
            torch.nn.GELU(),
            torch.nn.Linear(hidden_dim, VOCAB_SIZE),
        )

    def forward(self, x):
        x = self.emb_layer(x)
        x_pos = self.pos_emb_layer(torch.arange(0, x.shape[1]).to(DEVICE))
        x += x_pos
        x = self.attention_head(x)
        return self.layers(x)


basic_model_with_pos_atten = BasicModelWithPosEmbAttenHead().to(DEVICE)

In [96]:
test_text = "Suno Dost, Adeel"
test_text_encoded = tokenizer_gpt2.encode(test_text)
basic_model_with_pos.eval()
with torch.no_grad():
    out_logits = basic_model_with_pos(torch.tensor([test_text_encoded]).to(DEVICE))
next_token = torch.argmax(out_logits.squeeze()[-1])
tokenizer_gpt2.decode([next_token.item()])

' rang'

In [ ]:
optim = torch.optim.AdamW(basic_model_with_pos.parameters(), lr=0.01)
basic_model_with_pos.to(DEVICE)
results = []
for epoch in range(TOTAL_EPOCHS):
    progress_bar = tqdm(
        train_ds_loader,
        desc=f"Epoch {epoch + 1}/{TOTAL_EPOCHS}",
    )
    for ip, tar in progress_bar:
        optim.zero_grad()
        # print(ip.shape, tar.shape)
        out_logits = basic_model_with_pos(ip.to(DEVICE))
        loss = torch.nn.functional.cross_entropy(
            out_logits.transpose(1, 2), tar.to(DEVICE)
        )
        # print(loss)
        loss.backward()
        optim.step()
        progress_bar.set_postfix(loss=f"{loss:.4f}")
    results.append({"epoch": epoch, "loss": loss.item()})

In [ ]:
# from colab: Epoch 10/10: 100%|██████████| 65/65 [00:11<00:00,  5.78it/s, loss=0.0093]
# Output: Suno Dost, Shah Nawaz , jaltley ,pasaraaraaraaraaraaraaraaraaraaraaraaraara
# but it forgot its sentence strucutre and overfitted, time to introduce the validation loss